<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Task-Oriented Agent Workflow </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>


**Purpose:**  
Demonstrate how to build a **task-oriented agent** that can solve realistic, multi-step workflows by decomposing tasks, using tools, chaining actions, and knowing when the task is complete.

**Key Topics Covered Today:**
- Applying agents to real workflows
- **Task decomposition** — breaking big goals into smaller steps
- Agents interacting with external systems (tools)
- **Tool chaining** and combining retrieval + actions
- Autonomy control and safety
- Practical use cases: research, analysis, and simple coding agents

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" beautifulsoup4 requests duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/1

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.1 MB/s eta 0:00:00


## 2. Understanding Task-Oriented Agents

A **task-oriented agent** goes beyond simple Q&A. It can:
- Break down complex goals (task decomposition)
- Use multiple tools in sequence (tool chaining)
- Combine information retrieval with calculations/actions
- Decide when the task is finished (task completion criteria)
- Maintain some autonomy while staying safe

Today’s demo shows a realistic workflow: **Research + Analysis**.

## 3. Defining Tools

We will give the agent two powerful capabilities:
1. **Web search** — to retrieve real-world information
2. **Math calculator** — to perform calculations on the retrieved data

In [16]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform basic math calculations.
    Examples: '15 + 7', '(51.77 + 53.74) / 2', 'sum([10, 20, 30])'"""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [search_tool, calculate_math]
print(" Tools ready:", [t.name for t in tools])

 Tools ready: ['Web_Search', 'calculate_math']


## 4. Creating the ReAct Agent

We use LangChain’s `create_react_agent` for a simple but effective reasoning loop.

In [17]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langsmith import Client
client = Client()
llm = ChatOpenAI(model= 'gpt-4o-mini', temperature=0)
prompt = client.pull_prompt("hwchase17/react",
                            dangerously_pull_public_prompt=True)
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
excuter = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors = True,
    max_iteration=10
)
print(" Agent and Executor created")

 Agent and Executor created


In [18]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Pull a standard ReAct prompt
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,          # Shows full reasoning trace — great for learning!
    handle_parsing_errors=True,
    max_iterations=10      # Safety: prevent infinite loops
)

print(" Agent and Executor created")

 Agent and Executor created


## 5. Demo: Multi-Step Task Example

**Task:**  
"Find the current population of the top 3 most populous countries in the world in 2026. Calculate their combined population and then find what percentage the largest country represents of that total."

This task requires:
- Retrieval (web search)
- Data extraction
- Multiple math calculations (tool chaining)
- Clear final answer

# 1st

In [12]:
query = """Find the current population of the top 3 most populous province in the Saudi Arabia in 2026.
Output:
1- List the top 3 populous province in the Saudi Arabia
2- Calculate their combined population
3- Calculate exact percentage people that lives in these top 3 regions combined.
4- Calculate what percentage the largest Province represents of that total. """

response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to first find the current population of the top 3 most populous provinces in Saudi Arabia as of 2026. This will involve searching for the latest population data for these provinces.

Action: Web_Search  
Action Input: "current population of top 3 most populous provinces in Saudi Arabia 2026"  
26 Oct 2025 · It is the seat and largest city of Hail Province. As of 2022, Hail has a population of 498,575. Provinces of Saudi Arabia List 7 May 2026 · Saudi Arabia takes the 41st place in the world population rankings. The urban population in Saudi Arabia 2026 is 32.10 Million. Riyadh is the most populated ... 26 Apr 2026 · Other cities: Jeddah (gateway to Mecca, major port), Mecca (holiest city in Islam), Medina (second holiest city). Population: about 37 million (2025). Area: ~ ... 26 Apr 2026 · Ranked by population, from Riyadh and Jeddah to Mecca and Medina. Population: About 35 million. Population: about 37 million (20

# 2nd

In [6]:
query = """Find the current population of the top 3 most populous province in the Saudi Arabia in 2026.
Calculate their combined population and exact percentage of Saudi Arabia's total population that lives in these top 3 regions combined.
and then find what percentage the largest Province represents of that total. """

response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to first find the current population of the top 3 most populous provinces in Saudi Arabia as of 2026. I will then calculate their combined population and determine the percentage of people living in these provinces compared to the total population of Saudi Arabia. Finally, I will find out what percentage the largest province represents of that total.

Action: Web_Search  
Action Input: "current population of top 3 most populous provinces in Saudi Arabia 2026"  
1 week ago - This is a list of cities and towns in the Kingdom of Saudi Arabia. Saudi Arabia has five metropolitan areas with populations exceeding one million. ... ↑ "Saudi Arabia: Regions & Major Cities - Population Statistics, Maps, Charts, Weather and Web Information". 1 week ago - The provinces of Saudi Arabia, also referred to as regions, are the 13 first-level administrative divisions of the Kingdom of Saudi Arabia. Each province is governed by a gover

# best

In [ ]:
query = """Find the current population of the top 3 most populous province in the Saudi Arabia in 2026.
Calculate their combined population and then find what percentage the largest Province represents of that total."""

response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])

In [5]:
query = """Find the current population of the top 3 most populous province in the Saudi Arabia in 2026.
Calculate their combined population and then find what percentage the largest Province represents of that total."""

response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to first find the current population of the top 3 most populous provinces in Saudi Arabia as of 2026. This will involve searching for the latest population data for these provinces. After obtaining the populations, I will calculate their combined population and determine what percentage the largest province represents of that total.

Action: Web_Search  
Action Input: "current population of top 3 most populous provinces in Saudi Arabia 2026"  1 week ago - This is a list of cities and towns in the Kingdom of Saudi Arabia. Saudi Arabia has five metropolitan areas with populations exceeding one million. ... ↑ "Saudi Arabia: Regions & Major Cities - Population Statistics, Maps, Charts, Weather and Web Information". 3 days ago - Saudi Arabia is the seventh largest state in the Arab world, with a reported population of 35,300,280 as of 2024. 41.6% of inhabitants are immigrants. Saudi Arabia has experienced a population ex

# other idea

In [20]:
query = """Find the worldwide box office earnings of the top 2 highest-grossing movies of 2026.
Calculate the difference between their earnings, and then find the average revenue of these two movies."""
## best 2 movies with highest re
response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to first find the worldwide box office earnings of the top 2 highest-grossing movies of 2026. Since 2026 is in the future, I will perform a web search to see if there are any projections or current information available about these movies.

Action: Web_Search  
Action Input: "highest-grossing movies of 2026 box office earnings"  18 hours ago · Holmes Movie Trailer | Cinema: 2026 | Details ➤ https://KinoCheck.com/movie/46s/you-can-see-everything-2026 ... Top 20 Most Disturbing Movies Because of What We ... 17 hours ago · From housefull shows and strong box-office numbers to Twitter reviews praising Ravi Kishan, action, music and humour, the film is making its presence felt. But ... 3 days ago · lol. For sure. The Bird movie scares me! 1d. 2 days ago · To me, that was the biggest mistake that the 3rd movie did. Other than ... grossed $91.5M in China, 22.3% of its worldwide box office). Edit: Corrected ... 5 days ago ·

In [ ]:
query = """Find the current population of the top 3 most populous countries in the world in 2026.
Calculate their combined population and then find what percentage the largest country represents of that total."""

response = executor.invoke({"input": query})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to find the current population of the top 3 most populous countries in the world as of 2026. I will perform a web search to get the latest population figures for these countries. 

Action: Web_Search  
Action Input: "current population of the top 3 most populous countries in the world 2026"  1 day ago - It includes sovereign states, inhabited dependent territories and, in some cases, constituent countries of sovereign states, with inclusion within the list being primarily based on the ISO standard ISO 3166-1. For instance, the United Kingdom is considered a single entity, while the constituent countries of the Kingdom of the Netherlands are considered separately. In addition, this list includes certain states with limited recognition not found in ISO 3166-1. Also given in a percentage is each country's population compared with the world population, which the United Nations estimated at 8.232 billion as of 2025. 4 da

## 6. Another Practical Example — Research + Analysis Agent

Let’s try a different realistic workflow.

In [ ]:
research_query = """Research the latest developments in multimodal AI models in 2025-2026.
Summarize the top 2 key advancements, then calculate the average number of modalities mentioned across them (e.g. text, image, video, audio = 4)."""

response2 = executor.invoke({"input": research_query})
print("\n" + "="*50)
print("FINAL ANSWER:")
print(response2["output"])



> Entering new AgentExecutor chain...
I need to gather information on the latest developments in multimodal AI models for the years 2025-2026. This will involve searching for recent advancements and summarizing the top two key advancements. After that, I will calculate the average number of modalities mentioned in those advancements.

Action: Web_Search  
Action Input: "latest developments in multimodal AI models 2025 2026"  Top Takeaways 1. AI capability is not plateauing. It is accelerating and reaching more people than ever. Industry produced over 90% of notable frontier models in 2025, and several of those models now meet or exceed human baselines on PhD-level science questions, multimodal reasoning, and competition mathematics. The multimodal AI landscape evolves rapidly—models leading benchmarks today may trail new entrants within months as capabilities advance and pricing pressures intensify. Oct 9, 2025 · Explore the top multimodal AI models of 2026. Learn which works best fo

## 7. Task Decomposition in Action

Good agents naturally decompose tasks. Look at the **verbose trace** above — you should see steps like:

1. Thought: I need to search for current population data...
2. Action: Use Web_Search tool
3. Observation: Got data for India, China, USA...
4. Thought: Now I need to add the populations...
5. Action: Use calculate_math tool
6. ... and so on until Final Answer.

This is **tool chaining** and **multi-step reasoning**.

## 8. Autonomy Control and Safety

**Important Notes for Real-World Use:**

- We added `max_iterations=10` to prevent the agent from running forever.
- Always use `temperature=0` for more deterministic behavior in agents.
- For production, add human-in-the-loop approval for high-stakes actions (e.g., sending emails, making purchases).
- Monitor token usage and cost.

**Task Completion Criteria:**
The agent should stop when:
- All sub-tasks are done
- A clear, verifiable answer is reached
- No more tools are needed

You can improve completion by adding to the prompt:  
*"When the task is fully completed, respond with 'Final Answer:' followed by the result."*

## 9. Reflection Questions

1. How many tool calls did the agent make in the first demo? What was the sequence?
2. Where did you see **task decomposition** happening in the trace?
3. How does combining retrieval (search) and actions (math) make the agent more powerful?
4. What could go wrong if we remove the `max_iterations` limit?
5. Think of a real workflow in your domain (coding, research, business analysis). How would you turn it into an agent task?

---



You have now seen a **task-oriented agent** in action that can:
- Decompose realistic tasks
- Chain multiple tools
- Combine external information with calculations
- Operate with controlled autonomy
